# AG_PRAXIS NB06b — Saved Probabilities for Conformal Prediction

The sequence model in this project was trained once, scored once, and what it wrote down
was the class it picked for each window. That is all a per-class F1 needs. It is not all
every method needs. A conformal wrapper works from the whole score vector, because it
asks how confident the model was in every class and not only which one came out on top,
and those numbers were computed during scoring and then thrown away at the point where
the largest of them was taken.

This notebook gets them back. It loads the model that was already trained, runs it over
the validation windows and the test windows, and writes the full nineteen-column
probability matrix for each of them to disk, with the true labels beside it. Nothing is
trained. Nothing that already exists is touched. The model file is opened for reading,
and the arrays it reads were written by the preprocessing step and are not modified here.

The validation partition is the one that matters most, because it is the partition the
trained model has never seen and never been tuned on, which is what a conformal
calibration set has to be. It was not scored during training either, so this is the first
time those windows pass through the model at all. The test partition is included as well,
since a calibrated wrapper has to be evaluated on something, and only its labels were
saved before.

Two checks decide whether the output is trustworthy. The arrays have to arrive already
standardised, because the model was trained on standardised windows and would read raw
ones as nonsense while still returning a confident-looking answer. And the class picked
from these probabilities on the test partition has to match, window for window, the class
the original run recorded. If it does not, these are not the numbers that produced the
result on file, and nothing downstream should use them.

The data sits on Drive and the code sits in the repository, so the first block mounts one
and clones the other, and records the commit it is running from.

In [1]:
import os
import subprocess
import sys
from datetime import date
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AGREWAL14/AG_PRAXIS.git"
NOTEBOOK = "AG_PRAXIS_NB06b_cp_scores.ipynb"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO_ROOT = Path("/content/repo")
    if REPO_ROOT.exists():
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
else:
    REPO_ROOT = Path.cwd()
    while not (REPO_ROOT / "config" / "base.yaml").exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def git(*args):
    return subprocess.run(
        ["git", "-C", str(REPO_ROOT), *args], capture_output=True, text=True
    ).stdout.strip()


GIT_SHA = git("rev-parse", "--short", "HEAD")
GIT_BRANCH = git("rev-parse", "--abbrev-ref", "HEAD")
GIT_DIRTY = bool(git("status", "--porcelain"))
RUN_DATE = date.today().isoformat()

print(f"colab     : {IN_COLAB}")
print(f"repo root : {REPO_ROOT}")
print(f"git sha   : {GIT_SHA} on {GIT_BRANCH}" + ("   WORKING TREE DIRTY" if GIT_DIRTY else ""))
print(f"run date  : {RUN_DATE}")

Mounted at /content/drive
colab     : True
repo root : /content/repo
git sha   : 34f8ffd on main
run date  : 2026-08-10


The parameters come from `config/base.yaml`, and what the data looks like comes from the
manifest the preprocessing step wrote: which columns survived, which classes there are,
how many windows each class has in each partition, and the mean and scale of the
standardiser that was fitted on the training rows. Nothing is retyped from either.

Two of the paths below point at work that already exists and is only read. One is the
directory holding the trained sequence model and the labels it predicted. The other is
the directory holding the window arrays. Neither is written to. Output goes to a
directory of its own, so that a file written here cannot land on top of a file written
by the run that produced the model.

In [2]:
import json
import time

import numpy as np
import pandas as pd

from src import inventory as inv
from src import sequence as sq

CFG = inv.load_config(REPO_ROOT)

SEED = CFG["seed"]
WINDOW = int(CFG["sequence"]["window"])
STRIDE = int(CFG["sequence"]["stride"])
ARTIFACTS = Path(CFG["paths"]["artifacts"])

NB04_DIR = ARTIFACTS / "NB04"
NB06_RUN = "sequence_cnn_lstm_19class"
NB06_DIR = ARTIFACTS / "NB06" / NB06_RUN
OUT_DIR = (ARTIFACTS / "NB06b_cp_scores") if IN_COLAB else (REPO_ROOT / "results" / "NB06b_cp_scores")
PARTITIONS = ("val", "test")
PREDICT_BATCH = 512

if IN_COLAB and not ARTIFACTS.exists():
    raise FileNotFoundError(
        f"{ARTIFACTS} does not exist. Drive is not mounted, or the artefacts path in "
        "config/base.yaml is wrong. Nothing this notebook writes would survive."
    )


def first_existing(candidates, what):
    found = next((Path(p) for p in candidates if Path(p).exists()), None)
    if found is None:
        raise FileNotFoundError(f"{what} not found. Looked in: {[str(p) for p in candidates]}")
    return found


MANIFEST_PATH = first_existing(
    [REPO_ROOT / "data" / "processed" / "NB04_manifest.json", NB04_DIR / "NB04_manifest.json"],
    "NB04_manifest.json",
)
NB06_METRICS_PATH = first_existing(
    [REPO_ROOT / "data" / "processed" / "NB06" / "metrics.json", NB06_DIR / "metrics.json"],
    f"the metrics of {NB06_RUN}",
)

MANIFEST = json.loads(MANIFEST_PATH.read_text())
NB06_METRICS = json.loads(NB06_METRICS_PATH.read_text())

if MANIFEST.get("is_fast_pass"):
    raise ValueError(f"{MANIFEST_PATH} came from a fast pass and is not a result")

FEATURES = list(MANIFEST["columns"]["kept"])
CLASSES = list(NB06_METRICS["labels"])
SEQUENCES = {
    name: dict(MANIFEST["arrays"][f"sequences_{name}"]["by_class"]) for name in PARTITIONS
}
RAW_MEAN = MANIFEST["scaler"]["mean"]
RAW_SCALE = MANIFEST["scaler"]["scale"]

if sorted(CLASSES) != sorted(MANIFEST["arrays"]["sequences_val"]["by_class"]):
    raise ValueError("the model's label list and the manifest's classes are not the same set")

print(f"manifest    : {MANIFEST_PATH}")
print(f"model run   : {NB06_DIR}")
print(f"arrays from : {NB04_DIR}")
print(f"writing to  : {OUT_DIR}")
print()
print(f"{len(FEATURES)} features, {len(CLASSES)} classes, window {WINDOW}, stride {STRIDE}")
print(f"windows on record: val {sum(SEQUENCES['val'].values()):,}, "
      f"test {sum(SEQUENCES['test'].values()):,}")

manifest    : /content/repo/data/processed/NB04_manifest.json
model run   : /content/drive/MyDrive/AG_PRAXIS_artifacts/NB06/sequence_cnn_lstm_19class
arrays from : /content/drive/MyDrive/AG_PRAXIS_artifacts/NB04
writing to  : /content/drive/MyDrive/AG_PRAXIS_artifacts/NB06b_cp_scores

44 features, 19 classes, window 50, stride 25
windows on record: val 52,637, test 49,159


Nothing in this notebook is random. No model is built, no weights are initialised, no
sampling happens, and a forward pass over a trained network returns the same numbers
every time it is run. The seed is still set before anything else, because the rule in
this project is that the seed is set before any model exists in the session, and a rule
that is followed only when it appears to matter is not being followed.

In [3]:
import random

import keras

random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

print(f"seed {SEED} set, keras {keras.__version__}")

seed 42 set, keras 3.13.2


Now the model. It was saved whole, architecture and weights together, so it opens for
inference without being rebuilt and without any training code running. It is loaded
uncompiled, because a forward pass needs no optimiser and the optimiser state is not
wanted here.

Three things are checked against the file the original run wrote. The shape it expects
in, which fixes the window length and the number of columns. The width of what comes
out, which has to be one column per class. And the parameter count, which is the cheapest
way to confirm that the weights on disk are the network that produced the recorded
result rather than some other version of it.

In [4]:
MODEL_PATH = NB06_DIR / "model.keras"
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"{MODEL_PATH} is missing. This notebook reads the model the sequence run saved and "
        "does not train one. Check that Drive is mounted and that the run completed."
    )

model = keras.models.load_model(MODEL_PATH, compile=False)

expected_in = (WINDOW, len(FEATURES), 1)
got_in = tuple(int(v) for v in model.input_shape[1:])
got_out = int(model.output_shape[-1])
n_parameters = int(model.count_params())

print(f"loaded      : {MODEL_PATH.name} ({MODEL_PATH.stat().st_size / 1e6:.1f} MB)")
print(f"input shape : {got_in}, expected {expected_in}")
print(f"output width: {got_out}, expected {len(CLASSES)}")
print(f"parameters  : {n_parameters:,}, metrics.json records {NB06_METRICS['n_parameters']:,}")

assert got_in == expected_in, f"the model takes {got_in}, not the {expected_in} windows on disk"
assert got_out == len(CLASSES), f"the model has {got_out} outputs for {len(CLASSES)} classes"
assert n_parameters == int(NB06_METRICS["n_parameters"]), (
    "the loaded weights are a different network than the one metrics.json was written from"
)

loaded      : model.keras (2.6 MB)
input shape : (50, 44, 1), expected (50, 44, 1)
output width: 19, expected 19
parameters  : 214,227, metrics.json records 214,227


The windows come next. The preprocessing step wrote each partition as one compressed
file holding the windows, their labels, the column names, the class names, and the
window and stride they were cut at. Each of those is checked against the manifest on the
way in, so a file cut differently cannot be read as if it were this one.

Then the question that has to be settled before predicting: are these arrays already
standardised. The model was fitted on standardised windows. Feeding it raw ones would
not raise anything, it would return a full set of confident probabilities computed from
inputs the network has never seen the scale of, and every number after that would be
wrong in a way nothing downstream could detect.

The manifest records the mean of each column before standardising. Seven of the
forty-four have a raw mean above a hundred, and one of those is above eighty million.
Those are the columns where the two possibilities are far apart: if the arrays are
standardised their observed means sit near zero, and if they are raw their observed means
sit near the recorded ones. The check below reads the observed mean of every column off
the validation windows and requires those seven to be at least a hundred times closer to
zero than to their raw value.

In [5]:
def read_partition(name):
    """One partition of windows, checked against the manifest on the way in."""
    path = NB04_DIR / f"sequences_{name}.npz"
    if not path.exists():
        raise FileNotFoundError(f"{path} is missing. The preprocessing step writes it.")
    with np.load(path, allow_pickle=False) as npz:
        if [str(v) for v in npz["features"]] != FEATURES:
            raise ValueError(f"{path.name} holds different columns than the manifest lists")
        if [str(v) for v in npz["classes"]] != CLASSES:
            raise ValueError(f"{path.name} holds different classes than the model's labels")
        if (int(npz["window"]), int(npz["stride"])) != (WINDOW, STRIDE):
            raise ValueError(f"{path.name} was cut at a different window or stride")
        X, y = npz["X"], npz["y"].astype("int64")
    if X.shape[1:] != (WINDOW, len(FEATURES)):
        raise ValueError(f"{path.name} holds {X.shape[1:]} windows, not {(WINDOW, len(FEATURES))}")
    if not np.isfinite(X).all():
        raise ValueError(f"{path.name} holds a value that is not finite")
    counted = {c: int(n) for c, n in zip(CLASSES, np.bincount(y, minlength=len(CLASSES)))}
    if counted != SEQUENCES[name]:
        raise ValueError(f"{path.name} holds different per-class counts than the manifest records")
    print(f"  sequences_{name}.npz  {str(X.shape):>22}  {X.dtype}  {len(y):,} labels")
    return X, y


RAW = {}
for name in PARTITIONS:
    RAW[name] = read_partition(name)

flat = RAW["val"][0].reshape(-1, len(FEATURES))
observed_mean = flat.mean(axis=0, dtype="float64")
observed_std = flat.std(axis=0, dtype="float64")

scaling = pd.DataFrame(
    {
        "feature": FEATURES,
        "raw_mean": [float(RAW_MEAN[f]) for f in FEATURES],
        "raw_scale": [float(RAW_SCALE[f]) for f in FEATURES],
        "observed_mean": observed_mean,
        "observed_std": observed_std,
    }
)
for column in ("raw_mean", "raw_scale", "observed_mean", "observed_std"):
    if not pd.api.types.is_numeric_dtype(scaling[column]):
        raise TypeError(f"{column} came out as {scaling[column].dtype}, not numeric")

loud = scaling[scaling["raw_mean"].abs() > 100].copy()
loud["times_closer_to_zero"] = loud["raw_mean"].abs() / loud["observed_mean"].abs().clip(lower=1e-12)

print()
print("the columns with a large raw mean, as they arrive")
print(loud[["feature", "raw_mean", "observed_mean", "observed_std", "times_closer_to_zero"]]
      .sort_values("raw_mean", ascending=False)
      .to_string(index=False, float_format=lambda v: f"{v:,.4f}"))

still_raw = loud[loud["times_closer_to_zero"] < 100]
assert still_raw.empty, (
    "these columns arrive at their unstandardised scale, so the arrays have not been "
    f"standardised and the model must not be run on them: {list(still_raw['feature'])}"
)
print()
print(f"all {len(loud)} loud columns sit near zero, so the arrays carry the training "
      "standardisation already and none is applied here")

DATA = {name: {"X": sq.reshape(RAW[name][0]), "y": RAW[name][1]} for name in PARTITIONS}
for name in PARTITIONS:
    print(f"  {name:<5} fed to the model as {DATA[name]['X'].shape}")

  sequences_val.npz         (52637, 50, 44)  float32  52,637 labels
  sequences_test.npz         (49159, 50, 44)  float32  49,159 labels

the columns with a large raw mean, as they arrive
      feature        raw_mean  observed_mean  observed_std  times_closer_to_zero
          IAT 84,684,530.1803        -0.0001        1.0030  786,391,376,498.2188
Header_Length     28,266.0307        -0.0030        0.9952        9,298,041.8333
         Rate     15,924.3439        -0.0321        0.9571          496,755.8138
        Srate     15,924.3439        -0.0321        0.9571          496,755.8138
   Covariance      2,476.2055        -0.0006        0.9795        4,072,007.3120
      Tot sum        628.4982         0.0340        1.1415           18,473.0581
       Weight        141.5274         0.0001        1.0029        2,180,721.9110

all 7 loud columns sit near zero, so the arrays carry the training standardisation already and none is applied here
  val   fed to the model as (52637, 50, 44, 1)


The forward pass, and the writing, in one cell. Both partitions are predicted and all
five files are written before anything is checked, so that a check which fails does not
also throw away the work. The probabilities are stored as float32, which is what comes
out of the network and is enough for a score, and the labels as int8 positions into the
class list, which is how the rest of this project stores labels. Four arrays and a small
file recording what they are and where they came from.

In [6]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

PROBS, SECONDS = {}, {}
for name in PARTITIONS:
    started = time.perf_counter()
    PROBS[name] = np.asarray(
        model.predict(DATA[name]["X"], batch_size=PREDICT_BATCH, verbose=0), dtype="float32"
    )
    SECONDS[name] = round(time.perf_counter() - started, 3)
    print(f"{name:<5} {PROBS[name].shape} in {SECONDS[name]:.1f}s "
          f"({len(PROBS[name]) / max(SECONDS[name], 1e-9):,.0f} windows/s)")

written = {}
for name in PARTITIONS:
    np.save(OUT_DIR / f"probs_{name}.npy", PROBS[name])
    np.save(OUT_DIR / f"y_true_{name}.npy", DATA[name]["y"].astype("int8"))
    for stem in (f"probs_{name}", f"y_true_{name}"):
        written[f"{stem}.npy"] = (OUT_DIR / f"{stem}.npy").stat().st_size

scores = {
    "written_by": NOTEBOOK,
    "run_date": RUN_DATE,
    "git_sha": GIT_SHA + (" (working tree dirty)" if GIT_DIRTY else ""),
    "seed": SEED,
    "what": (
        "the full softmax matrix of the trained sequence model on the validation and test "
        "windows, with the true labels beside it. No model was trained here."
    ),
    "model_read_from": str(MODEL_PATH),
    "model_run_id": NB06_RUN,
    "n_parameters": n_parameters,
    "arrays_read_from": str(NB04_DIR),
    "window": WINDOW,
    "stride": STRIDE,
    "n_features": len(FEATURES),
    "labels": CLASSES,
    "label_encoding": "y_true_*.npy hold int8 positions into the labels list above",
    "probability_dtype": "float32, one row per window, one column per class, in label order",
    "partitions": {
        name: {
            "n_windows": int(len(PROBS[name])),
            "probs_file": f"probs_{name}.npy",
            "labels_file": f"y_true_{name}.npy",
            "inference_seconds": SECONDS[name],
            "by_class": SEQUENCES[name],
        }
        for name in PARTITIONS
    },
    "files": {k: int(v) for k, v in written.items()},
    "not_standardised_here": (
        "the window arrays already carry the standardisation fitted on the training rows in "
        "preprocessing, checked against the recorded column means before predicting"
    ),
}
(OUT_DIR / "scores.json").write_text(json.dumps(scores, indent=2, default=str) + "\n")

print()
print(f"written to {OUT_DIR}")
for filename, size in sorted(written.items()):
    print(f"  {filename:<20} {size / 1e6:>8.2f} MB")
print(f"  {'scores.json':<20} {(OUT_DIR / 'scores.json').stat().st_size / 1e6:>8.2f} MB")

val   (52637, 19) in 42.1s (1,249 windows/s)
test  (49159, 19) in 41.8s (1,176 windows/s)

written to /content/drive/MyDrive/AG_PRAXIS_artifacts/NB06b_cp_scores
  probs_test.npy           3.74 MB
  probs_val.npy            4.00 MB
  y_true_test.npy          0.05 MB
  y_true_val.npy           0.05 MB
  scores.json              0.00 MB


Now the check that decides whether any of this is usable. The original run saved the
class it picked for every test window. Taking the largest probability of each row of the
test matrix has to give back exactly that, window for window, because it is the same
operation on the same numbers from the same weights. Agreement below one means the
matrix on disk is not the one that produced the result on file.

The true labels are checked the same way, which catches the quieter failure: an array
read in a different order would still argmax to something, and comparing the labels
against the ones the run recorded is what rules that out.

The validation partition has nothing to compare against, since the run never scored it.
What can be checked there is that every row is a probability distribution, and that the
number of windows of each class is what the manifest says it should be, which is the
count a per-class calibration would have to work with.

In [7]:
banked_pred = np.load(NB06_DIR / "y_pred.npy")
banked_true = np.load(NB06_DIR / "y_true.npy")

argmax_test = PROBS["test"].argmax(axis=1).astype("int8")
agreement = float((argmax_test == banked_pred).mean())
disagreeing = int((argmax_test != banked_pred).sum())
labels_match = bool((DATA["test"]["y"].astype("int8") == banked_true).all())

print(f"test windows                    {len(argmax_test):,}")
print(f"argmax against the saved y_pred {agreement:.6f} agreement, {disagreeing} differ")
print(f"true labels against saved y_true {'identical' if labels_match else 'DIFFERENT'}")
print()

for name in PARTITIONS:
    row_sums = PROBS[name].sum(axis=1)
    print(f"{name:<5} row sums min {row_sums.min():.6f} max {row_sums.max():.6f}, "
          f"probabilities in [{PROBS[name].min():.2e}, {PROBS[name].max():.6f}]")
    assert np.allclose(row_sums, 1.0, atol=1e-4), f"{name} rows are not distributions"

assert labels_match, (
    "the test labels read here are not the labels the run scored, so the two are not aligned "
    "and nothing may be paired against the saved predictions"
)
assert disagreeing == 0, (
    f"{disagreeing} of {len(argmax_test):,} test windows argmax to a different class than the "
    "run recorded, so these probabilities did not produce the result on file"
)
print()
print("the saved probabilities reproduce the recorded predictions exactly")

test windows                    49,159
argmax against the saved y_pred 0.999797 agreement, 10 differ
true labels against saved y_true identical

val   row sums min 1.000000 max 1.000000, probabilities in [1.07e-19, 1.000000]
test  row sums min 1.000000 max 1.000000, probabilities in [9.20e-21, 1.000000]


AssertionError: 10 of 49,159 test windows argmax to a different class than the run recorded, so these probabilities did not produce the result on file

The last thing to look at is what a per-class calibration would actually have to work
with, which is the number of validation windows each class leaves. The rare classes are
the reason to look: a calibration set of a handful of items cannot support a tight
guarantee no matter how the arithmetic is arranged, and it is better to see the counts
now than to discover them inside a method that reports a number anyway.

In [ ]:
counts = pd.DataFrame(
    {
        "class": CLASSES,
        "val_windows": [SEQUENCES["val"][c] for c in CLASSES],
        "test_windows": [SEQUENCES["test"][c] for c in CLASSES],
    }
).sort_values("val_windows")
counts["val_share"] = counts["val_windows"] / counts["val_windows"].sum()

for column in ("val_windows", "test_windows", "val_share"):
    if not pd.api.types.is_numeric_dtype(counts[column]):
        raise TypeError(f"{column} came out as {counts[column].dtype}, not numeric")

observed_val = {c: int(n) for c, n in zip(CLASSES, np.bincount(DATA["val"]["y"],
                                                              minlength=len(CLASSES)))}
assert observed_val == SEQUENCES["val"], "the validation labels do not match the manifest counts"

print("validation windows per class, which is what a per-class calibration would draw on")
print(counts.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print()
print("smallest three: " + ", ".join(
    f"{row['class']} {int(row.val_windows)}" for _, row in counts.head(3).iterrows()
))
print(f"total {int(counts['val_windows'].sum()):,} validation windows across "
      f"{len(CLASSES)} classes")

The block below is the entry for the results ledger, printed rather than written so that
what goes into the file is read once before it goes in.

In [9]:
dirty_note = " (working tree dirty)" if GIT_DIRTY else ""
smallest = ", ".join(
    f"{row['class']} {int(row.val_windows)}" for _, row in counts.head(3).iterrows()
)

entry = f"""
### NB06b — cp_scores ({RUN_DATE})

| field | value |
|---|---|
| notebook | {NOTEBOOK} |
| git sha | {GIT_SHA}{dirty_note} |
| seed | {SEED} |
| what it is | a diagnostic that saves scores, not a run: no model was trained and no existing artefact was changed |
| model read | {NB06_RUN}, loaded from model.keras, {n_parameters:,} parameters |
| partitions scored | validation {len(PROBS['val']):,} windows, test {len(PROBS['test']):,} windows |
| written | probs_val.npy {written['probs_val.npy'] / 1e6:.1f} MB, probs_test.npy {written['probs_test.npy'] / 1e6:.1f} MB, y_true_val.npy, y_true_test.npy, scores.json |
| probability dtype | float32, one row per window, one column per class, in the label order in scores.json |
| standardisation | none applied here; the arrays carry the training standardisation, checked against the recorded column means before predicting |
| reproduces the run | argmax of probs_test against the saved y_pred: {agreement:.6f} agreement, {disagreeing} differing of {len(argmax_test):,} |
| labels aligned | test labels identical to the saved y_true: {labels_match} |
| inference seconds | val {SECONDS['val']:.1f}, test {SECONDS['test']:.1f} |
| smallest validation classes | {smallest} |
| artifacts | {OUT_DIR} |
| status | intermediate artefacts, consumed by later work, not a result |
"""

print(entry)

NameError: name 'counts' is not defined

In [8]:
import json
from pathlib import Path

import numpy as np

DRIVE = Path("/content/drive/MyDrive/AG_PRAXIS_artifacts")
LOCAL = Path.cwd()
while not (LOCAL / "config" / "base.yaml").exists() and LOCAL != LOCAL.parent:
    LOCAL = LOCAL.parent

CANDIDATES = (LOCAL / "results" / "NB06b_cp_scores", DRIVE / "NB06b_cp_scores")
SCORES_DIR = next((p for p in CANDIDATES if p.exists()), None)
if SCORES_DIR is None:
    raise FileNotFoundError(f"no score directory found. Looked in: {[str(p) for p in CANDIDATES]}")
NB06_DIR = DRIVE / "NB06" / "sequence_cnn_lstm_19class"
if not NB06_DIR.exists():
    NB06_DIR = SCORES_DIR  # y_pred/y_true copied alongside

probs = np.load(SCORES_DIR / "probs_test.npy")
y_true = np.load(SCORES_DIR / "y_true_test.npy").astype("int64")
banked = np.load(NB06_DIR / "y_pred.npy").astype("int64")
labels = json.loads((SCORES_DIR / "scores.json").read_text())["labels"]

new = probs.argmax(axis=1)
order = np.argsort(probs, axis=1)[:, ::-1]
top1 = probs[np.arange(len(probs)), order[:, 0]]
top2 = probs[np.arange(len(probs)), order[:, 1]]
margin = top1 - top2

idx = np.flatnonzero(new != banked)
print(f"{len(idx)} of {len(banked):,} windows disagree ({len(idx) / len(banked):.6%})")
print(f"agreement {1 - len(idx) / len(banked):.6f}")
print()

print(f"{'index':>8}  {'saved y_pred':<24}{'new argmax':<24}{'true label':<24}"
      f"{'p(saved)':>10}{'p(new)':>10}{'margin':>12}")
for i in idx:
    p_saved, p_new = float(probs[i, banked[i]]), float(probs[i, new[i]])
    print(f"{i:>8}  {labels[banked[i]]:<24}{labels[new[i]]:<24}{labels[y_true[i]]:<24}"
          f"{p_saved:>10.6f}{p_new:>10.6f}{abs(p_new - p_saved):>12.3e}")

m = margin[idx]
print()
print("margin between the top two classes on the disagreeing windows")
print(f"  min {m.min():.3e}   median {np.median(m):.3e}   max {m.max():.3e}")
for cut in (1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2):
    print(f"  below {cut:>8.0e}: {int((m < cut).sum())} of {len(m)}")

print()
print("for scale, the same margin over all test windows")
print(f"  median {np.median(margin):.4f}   "
      f"below 1e-2: {int((margin < 1e-2).sum()):,}   "
      f"below 1e-4: {int((margin < 1e-4).sum()):,}   "
      f"below 1e-6: {int((margin < 1e-6).sum()):,}")
at_risk = int((margin < 1e-5).sum())
print(f"  windows with margin below 1e-5: {at_risk:,}, of which {len(idx)} flipped")

print()
saved_right = int((banked[idx] == y_true[idx]).sum())
new_right = int((new[idx] == y_true[idx]).sum())
both_wrong = int(((banked[idx] != y_true[idx]) & (new[idx] != y_true[idx])).sum())
print(f"of the {len(idx)} disagreements: saved y_pred correct {saved_right}, "
      f"new argmax correct {new_right}, both wrong {both_wrong}")

print()
print("which classes the disagreements sit in")
for cls in sorted({labels[y_true[i]] for i in idx}):
    n = sum(1 for i in idx if labels[y_true[i]] == cls)
    print(f"  true {cls:<26} {n}")

print()
acc_new = float((new == y_true).mean())
acc_saved = float((banked == y_true).mean())
print(f"accuracy from these probabilities {acc_new:.6f}, from the saved y_pred {acc_saved:.6f}, "
      f"difference {abs(acc_new - acc_saved):.2e}")
try:
    from sklearn.metrics import f1_score

    print(f"macro-F1 from these probabilities {f1_score(y_true, new, average='macro'):.6f}, "
          f"from the saved y_pred {f1_score(y_true, banked, average='macro'):.6f}")
except ImportError:
    pass
print("(NB06 metrics.json records accuracy 0.8197 and macro-F1 0.7138)")

10 of 49,159 windows disagree (0.020342%)
agreement 0.999797

   index  saved y_pred            new argmax              true label                p(saved)    p(new)      margin
    2551  DoS-ICMP                DDoS-ICMP               DDoS-ICMP                 0.499721  0.499924   2.030e-04
    4282  DoS-ICMP                DDoS-ICMP               DDoS-ICMP                 0.498234  0.501675   3.441e-03
   19200  DoS-TCP                 DDoS-TCP                DDoS-TCP                  0.499696  0.500229   5.329e-04
   20939  DDoS-TCP                DoS-TCP                 DDoS-TCP                  0.499685  0.499920   2.344e-04
   21153  DoS-TCP                 DDoS-TCP                DDoS-TCP                  0.499382  0.500564   1.182e-03
   29683  DoS-ICMP                DDoS-ICMP               DoS-ICMP                  0.499611  0.499911   2.998e-04
   33036  DDoS-ICMP               DoS-ICMP                DoS-ICMP                  0.499799  0.500122   3.233e-04
   37540  DoS-TCP 